In [0]:

# Read existing Bronze data
df_bronze = spark.table(
    "workspace.bronze.citibike_station_status"
)

# Read existing Silver data
df_silver = spark.table(
    "workspace.silver.citibike_station_enriched"
)

# Find the unique Bronze snapshot IDs
bronze_snapshots = df_bronze.select(
    "snapshot_id"
).distinct()

# Find the unique Silver snapshot IDs
silver_snapshots = df_silver.select(
    "snapshot_id"
).distinct()

print("Bronze snapshots:", bronze_snapshots.count())
print("Silver snapshots:", silver_snapshots.count())


In [0]:

# Find Bronze snapshots missing from Silver
df_new_snapshots = bronze_snapshots.join(
    silver_snapshots,
    on="snapshot_id",
    how="left_anti"
)

# Count unprocessed snapshots
print(
    "Unprocessed snapshots:",
    df_new_snapshots.count()
)


In [0]:

# Count unprocessed Bronze snapshots
new_snapshot_count = df_new_snapshots.count()

print("New snapshots to process:", new_snapshot_count)

# Stop if there is no new data
if new_snapshot_count == 0:
    print("No new Bronze snapshots found.")
    dbutils.notebook.exit("NO_NEW_DATA")

# Continue if new data exists
print("New data found. Continuing Silver processing.")


In [0]:

# Select only unprocessed Bronze records
df_new_bronze = df_bronze.join(
    df_new_snapshots,
    on="snapshot_id",
    how="inner"
)

# Count the new records
print(
    "New Bronze records to process:",
    df_new_bronze.count()
)


In [0]:

from pyspark.sql import functions as F

df_new_silver = (
    df_new_bronze
    .select(
        "station_id",
        "num_bikes_available",
        "num_docks_available",
        "num_ebikes_available",
        "last_reported",
        "is_renting",
        "is_returning",
        "snapshot_id",
        "ingestion_timestamp"
    )
    .withColumnRenamed(
        "num_bikes_available", "bikes_available"
    )
    .withColumnRenamed(
        "num_docks_available", "docks_available"
    )
    .withColumnRenamed(
        "num_ebikes_available", "ebikes_available"
    )
    .withColumn(
        "last_reported_timestamp",
        F.timestamp_seconds(F.col("last_reported"))
    )
    .withColumn(
        "is_renting",
        F.col("is_renting").cast("boolean")
    )
    .withColumn(
        "is_returning",
        F.col("is_returning").cast("boolean")
    )
)

print("New Silver records:", df_new_silver.count())

display(df_new_silver.limit(5))


In [0]:

from pyspark.sql import functions as F

# Check missing station IDs
missing_station_ids = df_new_silver.filter(
    F.col("station_id").isNull() |
    (F.trim(F.col("station_id")) == "")
).count()

# Check missing availability values
missing_bike_counts = df_new_silver.filter(
    F.col("bikes_available").isNull() |
    F.col("docks_available").isNull() |
    F.col("ebikes_available").isNull()
).count()

# Check negative values
negative_values = df_new_silver.filter(
    (F.col("bikes_available") < 0) |
    (F.col("docks_available") < 0) |
    (F.col("ebikes_available") < 0)
).count()

# Check duplicate stations within each snapshot
duplicate_stations = (
    df_new_silver
    .groupBy("snapshot_id", "station_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Missing station IDs:", missing_station_ids)
print("Missing availability counts:", missing_bike_counts)
print("Negative value records:", negative_values)
print("Duplicate station groups:", duplicate_stations)


In [0]:

import requests

# Citi Bike Station Information API
info_url = "https://gbfs.citibikenyc.com/gbfs/en/station_information.json"

# Request station information
response_info = requests.get(info_url, timeout=20)

# Check for API errors
response_info.raise_for_status()

# Convert JSON response to Python data
info_data = response_info.json()

# Extract station information
station_info = info_data["data"]["stations"]

print("Station information records:", len(station_info))


In [0]:

# Prepare station metadata
station_info_records = []

for station in station_info:
    station_info_records.append({
        "station_id": station["station_id"],
        "station_name": station["name"],
        "latitude": station["lat"],
        "longitude": station["lon"],
        "capacity": station["capacity"],
        "station_type": station.get("station_type")
    })

# Convert metadata into a PySpark DataFrame
df_station_info = spark.createDataFrame(
    station_info_records
)

print(
    "Total station information records:",
    df_station_info.count()
)


In [0]:

# Join new Silver records with station information
df_new_enriched = df_new_silver.join(
    df_station_info,
    on="station_id",
    how="left"
)

# Compare records before and after joining
rows_before = df_new_silver.count()
rows_after = df_new_enriched.count()

print("Before join:", rows_before)
print("After join:", rows_after)


In [0]:

from pyspark.sql import functions as F

# Calculate bike availability percentage
df_new_enriched = df_new_enriched.withColumn(
    "availability_percentage",
    F.when(
        F.col("capacity") > 0,
        (F.col("bikes_available") / F.col("capacity")) * 100
    ).otherwise(None)
)

print("Availability percentage calculation completed.")


In [0]:

from pyspark.sql import functions as F

# Check missing station information
missing_station_info = df_new_enriched.filter(
    F.col("station_name").isNull()
).count()

# Check duplicate station records
duplicate_groups = (
    df_new_enriched
    .groupBy("snapshot_id", "station_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check invalid availability calculations
invalid_percentages = df_new_enriched.filter(
    (
        F.col("capacity").isNull() |
        (F.col("capacity") <= 0)
    ) &
    F.col("availability_percentage").isNotNull()
).count()

print("Rows before join:", rows_before)
print("Rows after join:", rows_after)
print("Missing station information:", missing_station_info)
print("Duplicate station groups:", duplicate_groups)
print("Invalid availability calculations:", invalid_percentages)


In [0]:

# Calculate whether all data quality checks passed
validation_passed = (
    rows_before == rows_after
    and missing_station_info == 0
    and duplicate_groups == 0
    and invalid_percentages == 0
    and missing_station_ids == 0
    and missing_bike_counts == 0
    and negative_values == 0
    and duplicate_stations == 0
)

# Stop the pipeline if validation fails
if not validation_passed:
    raise ValueError("Silver data quality validation failed!")

print("SUCCESS: All data quality checks passed!")


In [0]:

# Read the existing Silver Delta table
df_existing_silver = spark.table(
    "workspace.silver.citibike_station_enriched"
)

# Get the existing table schema
existing_schema = df_existing_silver.schema

# Check for missing required columns
missing_columns = (
    set(df_existing_silver.columns)
    - set(df_new_enriched.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required Silver columns: {missing_columns}"
    )

# Align new data to the existing Silver schema
df_ready_silver = df_new_enriched.select(
    *[
        F.col(field.name)
        .cast(field.dataType)
        .alias(field.name)
        for field in existing_schema.fields
    ]
)

print("Silver schema alignment completed.")
print("Columns:", len(df_ready_silver.columns))


In [0]:

# Compare prepared and existing Silver schemas
schema_matches = (
    df_ready_silver.schema == df_existing_silver.schema
)

print("Schema matches:", schema_matches)

# Stop processing if schemas do not match
if not schema_matches:
    raise ValueError("Silver schema mismatch!")

print("Silver data is ready for incremental ingestion!")


In [0]:

from delta.tables import DeltaTable

# Identify our existing Silver Delta table
silver_table = DeltaTable.forName(
    spark,
    "workspace.silver.citibike_station_enriched"
)

# Count records before merging
rows_before_merge = silver_table.toDF().count()

# Insert only records that don't already exist
(
    silver_table.alias("target")
    .merge(
        df_ready_silver.alias("source"),
        """
        target.snapshot_id = source.snapshot_id
        AND target.station_id = source.station_id
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# Verify the result
rows_after_merge = silver_table.toDF().count()

print("Rows before MERGE:", rows_before_merge)
print("Rows after MERGE:", rows_after_merge)
print("New rows inserted:", rows_after_merge - rows_before_merge)


In [0]:

from pyspark.sql import functions as F

# Read updated Silver table
df_verify = spark.table(
    "workspace.silver.citibike_station_enriched"
)

# Find duplicate station records
df_duplicates = (
    df_verify
    .groupBy("snapshot_id", "station_id")
    .count()
    .filter(F.col("count") > 1)
)

# Count duplicates
duplicate_count = df_duplicates.count()

print("Total Silver records:", df_verify.count())
print("Duplicate station groups:", duplicate_count)

# Stop the job if duplicates exist
if duplicate_count > 0:
    raise ValueError(
        "Duplicate station records found in Silver!"
    )

print("SUCCESS: Silver MERGE verification passed!")


In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT snapshot_id) AS total_snapshots,
    MAX(ingestion_timestamp) AS latest_ingestion
FROM workspace.silver.citibike_station_enriched;
